# Processamento Streaming — Dados de Alfabetização

Este notebook implementa a etapa de Streaming da arquitetura híbrida do projeto.

O objetivo é simular a chegada incremental de novos registros de avaliações de alunos e processá-los utilizando Spark Structured Streaming.

O fluxo demonstra como novos dados poderiam ser ingeridos continuamente, coexistindo com a pipeline Batch já implementada.

Fluxo:

Eventos de avaliação → Bronze Streaming → Silver Streaming

A camada Gold permanece responsável pelas análises consolidadas executadas em Batch.

## 1. Cenário de Streaming

O cenário simula novos registros de avaliações de alunos chegando de forma incremental.

Cada evento representa uma avaliação recebida pelo sistema educacional, contendo informações como ano, município, escola, aluno, presença, preenchimento da avaliação, situação de alfabetização e proficiência.

Em um ambiente produtivo, esses eventos poderiam ser provenientes de sistemas educacionais ou plataformas de aplicação das avaliações. Neste projeto, a chegada dos eventos será simulada para demonstrar o processamento com Spark Structured Streaming.

## 2. Fonte e estrutura dos eventos

Para demonstrar a ingestão incremental, serão utilizados eventos simulados de avaliações de alunos em formato JSON.

Cada novo arquivo representa um conjunto de eventos disponibilizados para processamento. O Spark Structured Streaming monitora continuamente o diretório de entrada e identifica automaticamente novos arquivos.

Essa abordagem permite demonstrar o comportamento de uma fonte incremental sem depender de serviços externos de mensageria, mantendo o foco no processamento com Spark Structured Streaming.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType
)

In [0]:
# Define os caminhos utilizados pelo pipeline de Streaming.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

STREAM_SOURCE_PATH = f"s3://{BUCKET_NAME}/streaming/source/"

BRONZE_STREAM_PATH = f"s3://{BUCKET_NAME}/bronze/streaming/avaliacao_alunos/"

BRONZE_CHECKPOINT_PATH = (
    f"s3://{BUCKET_NAME}/checkpoints/bronze_streaming/"
)

print(f"Source: {STREAM_SOURCE_PATH}")
print(f"Bronze Streaming: {BRONZE_STREAM_PATH}")
print(f"Checkpoint: {BRONZE_CHECKPOINT_PATH}")

In [0]:
# Define o schema dos eventos de avaliação recebidos pelo Streaming.

schema_evento = StructType([
    StructField("ano", IntegerType(), True),
    StructField("id_municipio", IntegerType(), True),
    StructField("id_escola", IntegerType(), True),
    StructField("id_aluno", IntegerType(), True),
    StructField("caderno", IntegerType(), True),
    StructField("serie", IntegerType(), True),
    StructField("rede", IntegerType(), True),
    StructField("presenca", IntegerType(), True),
    StructField("preenchimento_caderno", IntegerType(), True),
    StructField("alfabetizado", IntegerType(), True),
    StructField("proficiencia", DoubleType(), True),
    StructField("peso_aluno", DoubleType(), True)
])

## 3. Simulação dos eventos

Para simular a chegada incremental de dados, serão gerados pequenos lotes de eventos em formato JSON.

Cada lote representa novos registros de avaliações de alunos disponibilizados ao longo do tempo. Esses arquivos serão gravados no diretório de origem monitorado pelo Spark Structured Streaming.

In [0]:
# Cria o primeiro lote de eventos simulados de avaliações de alunos.

eventos_lote_1 = [
    {
        "ano": 2025,
        "id_municipio": 3550308,
        "id_escola": 60030001,
        "id_aluno": 99000001,
        "caderno": 1,
        "serie": 2,
        "rede": 3,
        "presenca": 1,
        "preenchimento_caderno": 1,
        "alfabetizado": 1,
        "proficiencia": 780.5,
        "peso_aluno": 1.0
    },
    {
        "ano": 2025,
        "id_municipio": 3550308,
        "id_escola": 60030001,
        "id_aluno": 99000002,
        "caderno": 2,
        "serie": 2,
        "rede": 3,
        "presenca": 1,
        "preenchimento_caderno": 1,
        "alfabetizado": 0,
        "proficiencia": 690.2,
        "peso_aluno": 1.0
    }
]

df_lote_1 = spark.createDataFrame(eventos_lote_1, schema=schema_evento)

display(df_lote_1)

In [0]:
# Grava o primeiro lote de eventos no diretório monitorado pelo Streaming.

LOTE_1_PATH = f"{STREAM_SOURCE_PATH}lote_1/"

(
    df_lote_1
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(LOTE_1_PATH)
)

print(f"[OK] Primeiro lote criado em: {LOTE_1_PATH}")

## 4. Bronze Streaming

Nesta etapa, o Spark Structured Streaming monitora o diretório de origem e processa novos arquivos JSON conforme são disponibilizados.

Os eventos são persistidos na camada Bronze Streaming sem aplicação de regras de negócio, preservando os dados recebidos e registrando o progresso por meio de checkpoint.

In [0]:
# Cria a leitura incremental dos eventos JSON, incluindo os arquivos armazenados nas subpastas de cada lote.

df_stream_bronze = (
    spark.readStream
    .option("recursiveFileLookup", "true")
    .schema(schema_evento)
    .json(STREAM_SOURCE_PATH)
)

print(f"Streaming ativo: {df_stream_bronze.isStreaming}")

In [0]:
# Persiste os eventos recebidos na Bronze Streaming em formato Parquet, utilizando checkpoint para controlar quais arquivos já foram processados.

query_bronze = (
    df_stream_bronze.writeStream
    .format("parquet")
    .option("path", BRONZE_STREAM_PATH)
    .option("checkpointLocation", BRONZE_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

query_bronze.awaitTermination()

print("[OK] Processamento Bronze Streaming concluído.")

In [0]:
# Relê os eventos persistidos na Bronze Streaming para validar a ingestão.

df_bronze_streaming = spark.read.parquet(BRONZE_STREAM_PATH)

print(
    f"[OK] Bronze Streaming: "
    f"{df_bronze_streaming.count()} registros"
)

display(df_bronze_streaming)

## 5. Validação da ingestão incremental

Após o processamento do primeiro lote, um segundo conjunto de eventos é adicionado à origem.

A nova execução do Structured Streaming deve processar apenas os arquivos ainda não registrados no checkpoint, demonstrando o comportamento incremental da pipeline.

In [0]:
# Cria o segundo lote de eventos simulados para validar o processamento incremental.

eventos_lote_2 = [
    {
        "ano": 2025,
        "id_municipio": 3550308,
        "id_escola": 60030002,
        "id_aluno": 99000003,
        "caderno": 3,
        "serie": 2,
        "rede": 3,
        "presenca": 1,
        "preenchimento_caderno": 1,
        "alfabetizado": 1,
        "proficiencia": 805.3,
        "peso_aluno": 1.0
    },
    {
        "ano": 2025,
        "id_municipio": 3550308,
        "id_escola": 60030002,
        "id_aluno": 99000004,
        "caderno": 4,
        "serie": 2,
        "rede": 3,
        "presenca": 0,
        "preenchimento_caderno": 0,
        "alfabetizado": 0,
        "proficiencia": None,
        "peso_aluno": None
    }
]

df_lote_2 = spark.createDataFrame(
    eventos_lote_2,
    schema=schema_evento
)

display(df_lote_2)

In [0]:
# Grava o segundo lote no diretório monitorado pelo Structured Streaming.

LOTE_2_PATH = f"{STREAM_SOURCE_PATH}lote_2/"

(
    df_lote_2
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(LOTE_2_PATH)
)

print(f"[OK] Segundo lote criado em: {LOTE_2_PATH}")

In [0]:
# Valida a quantidade total de eventos após o processamento incremental.

df_bronze_streaming = spark.read.parquet(BRONZE_STREAM_PATH)

print(
    f"[OK] Bronze Streaming após lote 2: "
    f"{df_bronze_streaming.count()} registros"
)

display(
    df_bronze_streaming.orderBy("id_aluno")
)

## 6. Silver Streaming

Nesta etapa, os eventos persistidos na Bronze Streaming são processados e validados antes de serem armazenados na camada Silver.

São aplicadas regras simples de qualidade e padronização compatíveis com as regras utilizadas na pipeline Batch, demonstrando como dados incrementais podem seguir o mesmo padrão de qualidade da arquitetura.

In [0]:
# Define os caminhos da Silver Streaming e do respectivo checkpoint.

SILVER_STREAM_PATH = (
    f"s3://{BUCKET_NAME}/silver/streaming/avaliacao_alunos/"
)

SILVER_CHECKPOINT_PATH = (
    f"s3://{BUCKET_NAME}/checkpoints/silver_streaming/"
)

print(f"Silver Streaming: {SILVER_STREAM_PATH}")
print(f"Checkpoint Silver: {SILVER_CHECKPOINT_PATH}")

In [0]:
# Cria a leitura incremental dos eventos persistidos na Bronze Streaming.

df_stream_silver = (
    spark.readStream
    .schema(schema_evento)
    .parquet(BRONZE_STREAM_PATH)
)

print(f"Streaming Silver ativo: {df_stream_silver.isStreaming}")

In [0]:
# Aplica regras de qualidade aos eventos antes da persistência na Silver.

from pyspark.sql import functions as F

df_stream_silver_tratado = (
    df_stream_silver

    # Mantém somente registros com identificadores essenciais.
    .filter(
        F.col("id_aluno").isNotNull()
        & F.col("id_municipio").isNotNull()
        & F.col("ano").isNotNull()
    )

    # Valida os valores categóricos esperados.
    .filter(F.col("presenca").isin(0, 1))
    .filter(F.col("preenchimento_caderno").isin(0, 1))
    .filter(F.col("alfabetizado").isin(0, 1))

    # Garante a regra observada na base histórica:
    # sem preenchimento válido, proficiência e peso permanecem nulos.
    .withColumn(
        "proficiencia",
        F.when(
            F.col("preenchimento_caderno") == 0,
            F.lit(None).cast("double")
        ).otherwise(F.col("proficiencia"))
    )

    .withColumn(
        "peso_aluno",
        F.when(
            F.col("preenchimento_caderno") == 0,
            F.lit(None).cast("double")
        ).otherwise(F.col("peso_aluno"))
    )
)

In [0]:
# Persiste os eventos tratados na Silver Streaming em formato Parquet, utilizando checkpoint independente da Bronze.

query_silver = (
    df_stream_silver_tratado.writeStream
    .format("parquet")
    .option("path", SILVER_STREAM_PATH)
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

query_silver.awaitTermination()

print("[OK] Processamento Silver Streaming concluído.")

In [0]:
# Relê os eventos persistidos na Silver Streaming para validar o processamento.

df_silver_streaming = spark.read.parquet(SILVER_STREAM_PATH)

print(
    f"[OK] Silver Streaming: "
    f"{df_silver_streaming.count()} registros"
)

display(
    df_silver_streaming.orderBy("id_aluno")
)

In [0]:
# Compara as quantidades de registros entre Bronze e Silver Streaming e valida as regras básicas de qualidade aplicadas.

registros_bronze = df_bronze_streaming.count()
registros_silver = df_silver_streaming.count()

invalidos_binarios = (
    df_silver_streaming
    .filter(
        (~F.col("presenca").isin(0, 1))
        | (~F.col("preenchimento_caderno").isin(0, 1))
        | (~F.col("alfabetizado").isin(0, 1))
    )
    .count()
)

inconsistencias_caderno = (
    df_silver_streaming
    .filter(
        (F.col("preenchimento_caderno") == 0)
        & (
            F.col("proficiencia").isNotNull()
            | F.col("peso_aluno").isNotNull()
        )
    )
    .count()
)

print(f"Bronze Streaming: {registros_bronze} registros")
print(f"Silver Streaming: {registros_silver} registros")
print(f"Valores binários inválidos: {invalidos_binarios}")
print(f"Inconsistências de caderno: {inconsistencias_caderno}")

if invalidos_binarios == 0 and inconsistencias_caderno == 0:
    print("[OK] Validações de qualidade do Streaming concluídas com sucesso.")

In [0]:
# Confirma quantos eventos existem fisicamente na origem do Streaming.

df_source_atual = (
    spark.read
    .option("recursiveFileLookup", "true")
    .schema(schema_evento)
    .json(STREAM_SOURCE_PATH)
)

print(f"Eventos disponíveis na origem: {df_source_atual.count()}")

display(
    df_source_atual.orderBy("id_aluno")
)

## 7. Validação final e metadados do Streaming

Nesta etapa são consolidadas as principais métricas da execução do pipeline de Streaming.

A validação confirma o processamento dos eventos entre Bronze e Silver, a aplicação das regras de qualidade e o uso de checkpoints para controle incremental.

In [0]:
# Consolida as métricas finais da execução do Streaming.

from datetime import datetime, timezone

data_hora_processamento = datetime.now(timezone.utc)

metricas_streaming = [{
    "pipeline": "avaliacao_alunos_streaming",
    "registros_bronze": df_bronze_streaming.count(),
    "registros_silver": df_silver_streaming.count(),
    "valores_binarios_invalidos": invalidos_binarios,
    "inconsistencias_caderno": inconsistencias_caderno,
    "formato_bronze": "parquet",
    "formato_silver": "parquet",
    "checkpoint": True,
    "status": "SUCESSO",
    "data_hora_processamento_utc": data_hora_processamento
}]

df_metricas_streaming = spark.createDataFrame(metricas_streaming)

display(df_metricas_streaming)

In [0]:
# Valida os critérios necessários para considerar o pipeline de Streaming executado com sucesso.

registro_bronze = df_bronze_streaming.count()
registro_silver = df_silver_streaming.count()

if registro_bronze != registro_silver:
    raise Exception(
        f"Quantidade divergente: Bronze={registro_bronze}, "
        f"Silver={registro_silver}"
    )

if invalidos_binarios > 0:
    raise Exception(
        f"Foram encontrados {invalidos_binarios} valores binários inválidos."
    )

if inconsistencias_caderno > 0:
    raise Exception(
        f"Foram encontradas {inconsistencias_caderno} inconsistências de caderno."
    )

print(
    "[OK] Streaming validado com sucesso: "
    f"{registro_bronze} eventos processados na Bronze e Silver, "
    "sem inconsistências de qualidade."
)

## 8. Resultado do processamento Streaming

A implementação demonstrou a ingestão e o processamento incremental de eventos de avaliação utilizando Spark Structured Streaming.

O primeiro lote foi processado e registrado por meio de checkpoint. Após a chegada de um segundo lote, apenas os novos eventos foram incorporados à pipeline, sem reprocessamento dos dados anteriores.

Os eventos passaram pelo fluxo:

**Fonte incremental → Bronze Streaming → Silver Streaming**

Na Silver foram aplicadas validações de campos obrigatórios, domínios binários e consistência entre preenchimento do caderno, proficiência e peso do aluno.

O fluxo demonstrou a coexistência entre processamento Batch e Streaming na arquitetura do projeto, mantendo os dados analíticos consolidados na Gold Batch.